In [1]:
#Task 4
# GPT-4'ün regex desenini ekle. Metni önce GPT-4'ün deseniyle parçalara
# böl, BPE'yi her parçada ayrı çalıştır. Tokenizer'ı yeniden eğit,
# birleştirmeleri öncekiyle karşılaştır. Sonra "Ankara'dan" ve
# "İstanbul'da" gibi kesme işaretli kelimeleri bu desenle böl. Desen
# İngilizce için yazılmış: Türkçede ne yapıyor, kendi cümlelerinle yaz.
# (0:57:36 - 1:11:38)

In [2]:
import regex as re
gpt4pat =r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
with open("altayli_kose_yazilari.txt", "r", encoding = "utf-8") as f:
    text = re.findall(gpt4pat, f.read())


tokens = [list(x.encode("utf-8")) for x in text]

In [3]:
#just like bigram week
def get_stats(ids):
    counts = {}
    for pair in zip(ids,ids[1:]):
        counts[pair] = counts.get(pair,0)+1
    return counts

In [4]:
def merge (ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair [1]:
            newids.append(idx)
            i += 2 
        else:
            newids.append(ids [i])
            i += 1
    return newids

In [5]:
mergesgpt4 = {}
ids = [list(chunk) for chunk in tokens]

for i in range(50):
    stats = {}
    for chunk in ids:
        stats_chunk = get_stats(chunk)
        for pair, count in stats_chunk.items():
            stats[pair] = stats.get(pair, 0) + count

    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into a new token {idx}")

    ids = [merge(chunk, pair, idx) for chunk in ids]
    mergesgpt4[pair] = idx

merging (196, 177) into a new token 256
merging (97, 114) into a new token 257
merging (195, 188) into a new token 258
merging (97, 110) into a new token 259
merging (32, 98) into a new token 260
merging (197, 159) into a new token 261
merging (10, 10) into a new token 262
merging (101, 114) into a new token 263
merging (105, 110) into a new token 264
merging (32, 100) into a new token 265
merging (44, 262) into a new token 266
merging (226, 128) into a new token 267
merging (256, 110) into a new token 268
merging (196, 159) into a new token 269
merging (101, 110) into a new token 270
merging (195, 167) into a new token 271
merging (32, 107) into a new token 272
merging (105, 114) into a new token 273
merging (32, 121) into a new token 274
merging (105, 108) into a new token 275
merging (109, 97) into a new token 276
merging (32, 103) into a new token 277
merging (111, 114) into a new token 278
merging (108, 257) into a new token 279
merging (195, 182) into a new token 280
merging (32,

In [6]:
print(re.findall(gpt4pat, "Ankara’dan"))
print(re.findall(gpt4pat, "Ankara'dan"))

['Ankara', '’dan']
['Ankara', "'d", 'an']


In [7]:
print(re.findall(gpt4pat,"Ankara'dan sabah erkenden çıktık, öğlene doğru İstanbul'daydık. Ayşe'nin annesi bizi Kadıköy'deki evinde bekliyordu."))

['Ankara', "'d", 'an', ' sabah', ' erkenden', ' çıktık', ',', ' öğlene', ' doğru', ' İstanbul', "'d", 'aydık', '.', ' Ayşe', "'nin", ' annesi', ' bizi', ' Kadıköy', "'d", 'eki', ' evinde', ' bekliyordu', '.']


In [8]:
print(re.findall(gpt4pat,"We left Ankara early in the morning, and by noon we'd reached Istanbul. Ayşe's mother was waiting for us at her house in Kadıköy."))

['We', ' left', ' Ankara', ' early', ' in', ' the', ' morning', ',', ' and', ' by', ' noon', ' we', "'d", ' reached', ' Istanbul', '.', ' Ayşe', "'s", ' mother', ' was', ' waiting', ' for', ' us', ' at', ' her', ' house', ' in', ' Kadıköy', '.']


In [9]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0,p1), idx in mergesgpt4.items():
    vocab[idx] = vocab[p0] + vocab[p1]

for idx in range(256, 306):
    print(idx, vocab[idx], vocab[idx].decode('utf-8', errors='replace'))

256 b'\xc4\xb1' ı
257 b'ar' ar
258 b'\xc3\xbc' ü
259 b'an' an
260 b' b'  b
261 b'\xc5\x9f' ş
262 b'\n\n' 


263 b'er' er
264 b'in' in
265 b' d'  d
266 b',\n\n' ,


267 b'\xe2\x80' �
268 b'\xc4\xb1n' ın
269 b'\xc4\x9f' ğ
270 b'en' en
271 b'\xc3\xa7' ç
272 b' k'  k
273 b'ir' ir
274 b' y'  y
275 b'il' il
276 b'ma' ma
277 b' g'  g
278 b'or' or
279 b'lar' lar
280 b'\xc3\xb6' ö
281 b' s'  s
282 b'ak' ak
283 b'iy' iy
284 b' o'  o
285 b'la' la
286 b'\xe2\x80\x99' ’
287 b'et' et
288 b'al' al
289 b'ek' ek
290 b' bir'  bir
291 b'el' el
292 b'un' un
293 b' i'  i
294 b'ay' ay
295 b' de'  de
296 b' h'  h
297 b'im' im
298 b' v'  v
299 b'da' da
300 b'de' de
301 b'd\xc4\xb1' dı
302 b' t'  t
303 b'di' di
304 b'at' at
305 b'as' as
